# Example 5: Weight constraints

This example deals with the use of basic weight and budget constraints.

In [1]:
using PortfolioOptimisers, PrettyTables
# Format for pretty tables.
tsfmt = (v, i, j) -> begin
    if j == 1
        return Date(v)
    else
        return v
    end
end;
resfmt = (v, i, j) -> begin
    if j == 1
        return v
    else
        return isa(v, Number) ? "$(round(v*100, digits=3)) %" : v
    end
end;
mipresfmt = (v, i, j) -> begin
    if j ∈ (1, 2, 3)
        return v
    else
        return isa(v, Number) ? "$(round(v*100, digits=3)) %" : v
    end
end;

## 1. Returns data

We will use the same data as the previous example.

In [2]:
using CSV, TimeSeries, DataFrames

X = TimeArray(CSV.File(joinpath(@__DIR__, "SP500.csv.gz")); timestamp = :Date)[(end - 252):end]
pretty_table(X[(end - 5):end]; formatters = tsfmt)

# Compute the returns
rd = prices_to_returns(X)

┌────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│  timestamp │    AAPL │     AMD │     BAC │     BBY │     CVX │      GE │      HD │     JNJ │     JPM │      KO │     LLY │     MRK │    MSFT │     PEP │     PFE │      PG │     RRC │     UNH │     WMT │     XOM │
│       Date │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │
├────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ 2022-12-20 │ 131.916 │   65.05 │  31.729 │  77.371 │ 169.497 │  62.604 │ 310.342 │ 173.109 │ 127.844 │  61.841 │  357.55 │ 108.229 │  240.

ReturnsResult
    nx | 20-element Vector{String}
     X | 252×20 Matrix{Float64}
    nf | nothing
     F | nothing
    ts | 252-element Vector{Date}
    iv | nothing
  ivpa | nothing


## 2. Preparatory steps

We'll provide a vector of continuous solvers beacause the optimisation type we'll be using is more complex, and will contain various constraints. We will also use a more exotic risk measure.

For the mixed interger solvers, we can use a single one.

In [3]:
using Clarabel, HiGHS
slv = [Solver(; name = :clarabel1, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel3, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.9),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel5, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.8),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel7, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.70),
              check_sol = (; allow_local = true, allow_almost = true))];
mip_slv = Solver(; name = :highs1, solver = HiGHS.Optimizer,
                 settings = Dict("log_to_console" => false),
                 check_sol = (; allow_local = true, allow_almost = true));

This time we will use the `EntropicValueatRisk` measure and we will once again precompute prior.

In [4]:
r = EntropicValueatRisk()
pr = prior(EmpiricalPriorEstimator(), rd)

LowOrderPriorResult
         X | 252×20 Matrix{Float64}
        mu | 20-element Vector{Float64}
     sigma | 20×20 Matrix{Float64}
      chol | nothing
         w | nothing
  loadings | nothing
      f_mu | nothing
   f_sigma | nothing
       f_w | nothing


## 3. Budget constraints

The `budget` is the value of the sum of a portfolio's weights.

Here we will showcase various budget constraints. We will start simple, with a strict budget constraint. We will also show the impact this has on the finite allocation.

### 3.1 Strict budget constraints

#### 3.1.1 Fully invested long only

First the default case, where the budget is equal to 1, `bgt = 1`. This means we are fully invested.

In [5]:
opt1 = JuMPOptimiser(; pe = pr, slv = slv)
mr1 = MeanRisk(; r = r, opt = opt1)

MeanRisk
  opt | JuMPOptimiser
      |       pe | LowOrderPriorResult
      |          |          X | 252×20 Matrix{Float64}
      |          |         mu | 20-element Vector{Float64}
      |          |      sigma | 20×20 Matrix{Float64}
      |          |       chol | nothing
      |          |          w | nothing
      |          |   loadings | nothing
      |          |       f_mu | nothing
      |          |    f_sigma | nothing
      |          |        f_w | nothing
      |      slv | Vector{Solver{Symbol, UnionAll, T3, @NamedTuple{allow_local::Bool, allow_almost::Bool}, Bool} where T3<:(Union{Nothing, var"#s234"} where var"#s234"<:AbstractDict)}: Solver{Symbol, UnionAll, T3, @NamedTuple{allow_local::Bool, allow_almost::Bool}, Bool} where T3<:(Union{Nothing, var"#s234"} where var"#s234"<:AbstractDict)[Solver
      |          name | Symbol: :clarabel1
      |        solver | UnionAll: Clarabel.MOIwrapper.Optimizer
      |      settings | Dict{String, Bool}: Dict{String, Bool}("ve

You can see that `wb` is of type `WeightBoundsResult`, `lb = 0.0` (asset weights lower bound), and `ub = 1.0` (asset weights upper bound), and the `bgt = 1.0` (budget).

We can check that the constraints were satisfied.

In [6]:
res1 = optimise!(mr1)
println("budget: $(sum(res1.w))")
println("long budget: $(sum(res1.w[res1.w .>= zero(eltype(res1.w))]))")
println("short budget: $(sum(res1.w[res1.w .< zero(eltype(res1.w))]))")
println("weight bounds: $(all(x -> zero(x) <= x <= one(x), res1.w))")

budget: 0.9999999998112562
long budget: 0.9999999998112562
short budget: 0.0
weight bounds: true


Now lets allocate a finite amount of capital, `4206.9`, to this portfolio.

In [7]:
da = DiscreteAllocation(; slv = mip_slv)
mip_res1 = optimise!(da, res1.w, vec(values(X[end])), 4206.9)
pretty_table(DataFrame(:assets => rd.nx, :shares => mip_res1.shares, :cost => mip_res1.cost,
                       :opt_weights => res1.w, :mip_weights => mip_res1.w);
             formatters = mipresfmt)
println("long cost + short cost = cost: $(sum(mip_res1.cost))")
println("long cost: $(sum(mip_res1.cost[mip_res1.cost .>= zero(eltype(mip_res1.cost))]))")
println("short cost: $(sum(mip_res1.cost[mip_res1.cost .< zero(eltype(mip_res1.cost))]))")
println("remaining cash: $(mip_res1.cash)")
println("used cash ≈ available cash: $(isapprox(sum(mip_res1.cost) + mip_res1.cash, 4206.9 * sum(res1.w)))")

┌────────┬─────────┬─────────┬─────────────┬─────────────┐
│ assets │  shares │    cost │ opt_weights │ mip_weights │
│ String │ Float64 │ Float64 │     Float64 │     Float64 │
├────────┼─────────┼─────────┼─────────────┼─────────────┤
│   AAPL │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    AMD │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    BAC │     1.0 │  32.301 │       0.0 % │     0.768 % │
│    BBY │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    CVX │     5.0 │  868.64 │    21.386 % │    20.655 % │
│     GE │     0.0 │     0.0 │       0.0 % │       0.0 % │
│     HD │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    JNJ │    13.0 │ 2263.11 │    55.414 % │    53.815 % │
│    JPM │     0.0 │     0.0 │       0.0 % │       0.0 % │
│     KO │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    LLY │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    MRK │     8.0 │ 876.648 │    21.207 % │    20.846 % │
│   MSFT │     0.0 │     0.0 │       0.0 % │       0.0 %

#### 3.1.2 Maximum risk-return ratio market neutral portfolio

Now lets create a maximum risk-return ratio market neutral portfolio. In order to do this we need to set the short budget and lower the weight bounds constraints. If we simply set the budget to `0`, the solver will return all zeros.

For a market neutral portfolio, the portfolio weights must sum to `0`. This means the long and short budgets must be equal but opposite in sign. We also need to remember to allow the weights lower bounds to be negative, otherwise we will get all zeros as well.

Lets do the simple case where the long and short budgets are `1`, and the weights bounds are `±1`.

Both The short budget `sbgt` has to be given as an absolute value (it simplifies implementation details), but the weight lower bounds can be negative.

Minimising the risk under these constraints without additional constraints often yields all zeros. So we will maximise the risk-return ratio.

In [8]:
rf = 4.2 / 100 / 252
opt2 = JuMPOptimiser(; pe = pr, slv = slv,
                     # Budget and short budget absolute values.
                     bgt = 0, sbgt = 1,
                     # Weight bounds.
                     wb = WeightBoundsResult(; lb = -1.0, ub = 1.0))
mr2 = MeanRisk(; r = r, obj = MaximumRatio(; rf = rf), opt = opt2)
res2 = optimise!(mr2)
println("budget: $(sum(res2.w))")
println("long budget: $(sum(res2.w[res2.w .>= zero(eltype(res2.w))]))")
println("short budget: $(sum(res2.w[res2.w .< zero(eltype(res2.w))]))")
println("weight bounds: $(all(x -> -one(x) <= x <= one(x), res2.w))")

budget: -1.0089443539131439e-16
long budget: 0.9999936641090608
short budget: -0.999993664109061
weight bounds: true


Lets see what happens when we allocate a finite amount of capital. Because we made the long and short budgets equal to 1, the entire available cash will be invested in both long and short positions.

There is a small variation due to the discrete nature of the allocation, but we can verify the statement above.

In [9]:
mip_res2 = optimise!(da, res2.w, vec(values(X[end])), 4206.9)
pretty_table(DataFrame(:assets => rd.nx, :shares => mip_res2.shares, :cost => mip_res2.cost,
                       :opt_weights => res2.w, :mip_weights => mip_res2.w);
             formatters = mipresfmt)
println("long cost + short cost = cost: $(sum(mip_res2.cost))")
println("long cost: $(sum(mip_res2.cost[mip_res2.cost .>= zero(eltype(mip_res2.cost))]))")
println("short cost: $(sum(mip_res2.cost[mip_res2.cost .< zero(eltype(mip_res2.cost))]))")
println("remaining cash: $(mip_res2.cash)")
println("used cash ≈ available cash: $(isapprox(sum(abs.(mip_res2.cost)) + mip_res2.cash, 4206.9 * sum(abs.(res2.w))))")

┌────────┬─────────┬──────────┬─────────────┬─────────────┐
│ assets │  shares │     cost │ opt_weights │ mip_weights │
│ String │ Float64 │  Float64 │     Float64 │     Float64 │
├────────┼─────────┼──────────┼─────────────┼─────────────┤
│   AAPL │    -1.0 │ -125.674 │    -2.526 % │    -2.999 % │
│    AMD │    -2.0 │  -125.14 │    -3.226 % │    -2.986 % │
│    BAC │   -21.0 │ -678.321 │   -16.732 % │   -16.188 % │
│    BBY │     1.0 │   78.279 │     0.837 % │     1.857 % │
│    CVX │     0.0 │      0.0 │     0.001 % │       0.0 % │
│     GE │     6.0 │  383.298 │     8.532 % │     9.093 % │
│     HD │     0.0 │     -0.0 │    -3.235 % │      -0.0 % │
│    JNJ │    -4.0 │  -696.34 │   -17.975 % │   -16.618 % │
│    JPM │     3.0 │  388.725 │     8.858 % │     9.222 % │
│     KO │    23.0 │  1440.01 │    33.949 % │    34.163 % │
│    LLY │     0.0 │      0.0 │     3.536 % │       0.0 % │
│    MRK │    16.0 │   1753.3 │    39.716 % │    41.595 % │
│   MSFT │    -1.0 │ -233.434 │    -7.63

#### 3.1.3 Short-only portfolio

We will now create and discretely allocate a short-only portfolio. This is in general an anti-pattern but oen can use various combinations of budget, weight bounds and short budget constraints to create hedging portfolios.

In [10]:
opt3 = JuMPOptimiser(; pe = pr, slv = slv,
                     # Budget and short budget absolute values.
                     bgt = -1, sbgt = 1,
                     # Weight bounds.
                     wb = WeightBoundsResult(; lb = -1.0, ub = 0.0))
mr3 = MeanRisk(; r = r, obj = MinimumRisk(), opt = opt3)
res3 = optimise!(mr3)
println("budget: $(sum(res3.w))")
println("long budget: $(sum(res3.w[res3.w .>= zero(eltype(res3.w))]))")
println("short budget: $(sum(res3.w[res3.w .< zero(eltype(res3.w))]))")
println("weight bounds: $(all(x -> -one(x) <= x <= zero(x), res3.w))")

mip_res3 = optimise!(da, res3.w, vec(values(X[end])), 4206.9)
pretty_table(DataFrame(:assets => rd.nx, :shares => mip_res3.shares, :cost => mip_res3.cost,
                       :opt_weights => res3.w, :mip_weights => mip_res3.w);
             formatters = mipresfmt)
println("long cost + short cost = cost: $(sum(mip_res3.cost))")
println("long cost: $(sum(mip_res3.cost[mip_res3.cost .>= zero(eltype(mip_res3.cost))]))")
println("short cost: $(sum(mip_res3.cost[mip_res3.cost .< zero(eltype(mip_res3.cost))]))")
println("remaining cash: $(mip_res3.cash)")
println("used cash ≈ available cash: $(isapprox(sum(mip_res3.cost) - mip_res3.cash, 4206.9 * sum(res3.w)))")

budget: -0.9999999995925778
long budget: 0.0
short budget: -0.9999999995925778
weight bounds: true
┌────────┬─────────┬──────────┬─────────────┬─────────────┐
│ assets │  shares │     cost │ opt_weights │ mip_weights │
│ String │ Float64 │  Float64 │     Float64 │     Float64 │
├────────┼─────────┼──────────┼─────────────┼─────────────┤
│   AAPL │     0.0 │     -0.0 │      -0.0 % │      -0.0 % │
│    AMD │     0.0 │     -0.0 │      -0.0 % │      -0.0 % │
│    BAC │    -1.0 │  -32.301 │      -0.0 % │    -0.768 % │
│    BBY │     0.0 │     -0.0 │      -0.0 % │      -0.0 % │
│    CVX │     0.0 │     -0.0 │      -0.0 % │      -0.0 % │
│     GE │    -3.0 │ -191.649 │    -4.404 % │    -4.558 % │
│     HD │    -1.0 │  -311.22 │    -5.068 % │    -7.401 % │
│    JNJ │    -2.0 │  -348.17 │     -9.44 % │     -8.28 % │
│    JPM │     0.0 │     -0.0 │      -0.0 % │      -0.0 % │
│     KO │     0.0 │     -0.0 │      -0.0 % │      -0.0 % │
│    LLY │     0.0 │     -0.0 │    -4.646 % │      -0.0 % │
│

#### 3.1.4 Over and under leveraged portfolios

The discrete allocation procedure automatically adjusts the cash amount depending on the optimal long and short weights, so there is no need to split the cash amount into long and short allocations. This means we can fearlessly under and overleverage the portfolio, and the discrete allocation will follow suit.

We can do this regardless of what combination of budget, short budget and weight bounds constraints we use. Lets try an overleveraged long-only portfolio and an underleveraged long-short portfolio.

Lets try the overleveraged long-only portfolio first.

In [11]:
opt4 = JuMPOptimiser(; pe = pr, slv = slv, bgt = 1.3)
mr4 = MeanRisk(; r = r, opt = opt4)
res4 = optimise!(mr4)
println("budget: $(sum(res4.w))")
println("long budget: $(sum(res4.w[res4.w .>= zero(eltype(res4.w))]))")
println("short budget: $(sum(res4.w[res4.w .< zero(eltype(res4.w))]))")
println("weight bounds: $(all(x -> zero(x) <= x <= one(x), res4.w))")

mip_res4 = optimise!(da, res4.w, vec(values(X[end])), 4206.9)
pretty_table(DataFrame(:assets => rd.nx, :shares => mip_res4.shares, :cost => mip_res4.cost,
                       :opt_weights => res4.w, :mip_weights => mip_res4.w);
             formatters = mipresfmt)
println("long cost + short cost = cost: $(sum(mip_res4.cost))")
println("long cost: $(sum(mip_res4.cost[mip_res4.cost .>= zero(eltype(mip_res4.cost))]))")
println("short cost: $(sum(mip_res4.cost[mip_res4.cost .< zero(eltype(mip_res4.cost))]))")
println("remaining cash: $(mip_res4.cash)")
println("used cash ≈ available cash: $(isapprox(sum(mip_res4.cost) + mip_res4.cash, 4206.9 * sum(res4.w)))")

budget: 1.2999999997828973
long budget: 1.2999999997828973
short budget: 0.0
weight bounds: true
┌────────┬─────────┬─────────┬─────────────┬─────────────┐
│ assets │  shares │    cost │ opt_weights │ mip_weights │
│ String │ Float64 │ Float64 │     Float64 │     Float64 │
├────────┼─────────┼─────────┼─────────────┼─────────────┤
│   AAPL │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    AMD │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    BAC │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    BBY │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    CVX │     5.0 │  868.64 │    27.802 % │    20.661 % │
│     GE │     0.0 │     0.0 │       0.0 % │       0.0 % │
│     HD │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    JNJ │    22.0 │ 3829.87 │    72.032 % │    91.094 % │
│    JPM │     0.0 │     0.0 │       0.0 % │       0.0 % │
│     KO │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    LLY │     0.0 │     0.0 │       0.0 % │       0.0 % │
│    MRK │     7.0

Lets try the underleveraged long-short portfolio next. Note that the short budget is not satisfied, that's because it is a relaxation constraint of the short weights. The the portfolio budget constraint satisfied, however. That is because it is an exact constraint. Later we will show how to set budget bounds constraint for the short budget as well as the portfolio budget. We will show these later.

We will be able to invest around half of our available cash (due to our finite resources), the rest will come from the short positions.

In [12]:
opt5 = JuMPOptimiser(; pe = pr, slv = slv,
                     # Budget and short budget absolute values.
                     bgt = 0.5, sbgt = 1,
                     # Weight bounds.
                     wb = WeightBoundsResult(; lb = -1.0, ub = 1.0))
mr5 = MeanRisk(; r = r, opt = opt5)
res5 = optimise!(mr5)
println("budget: $(sum(res5.w))")
println("long budget: $(sum(res5.w[res5.w .>= zero(eltype(res5.w))]))")
println("short budget: $(sum(res5.w[res5.w .< zero(eltype(res5.w))]))")
println("weight bounds: $(all(x -> -one(x) <= x <= one(x), res5.w))")

mip_res5 = optimise!(da, res5.w, vec(values(X[end])), 4506.9)
pretty_table(DataFrame(:assets => rd.nx, :shares => mip_res5.shares, :cost => mip_res5.cost,
                       :opt_weights => res5.w, :mip_weights => mip_res5.w);
             formatters = mipresfmt)
println("long cost + short cost = cost: $(sum(mip_res5.cost))")
println("long cost: $(sum(mip_res5.cost[mip_res5.cost .>= zero(eltype(mip_res5.cost))]))")
println("short cost: $(sum(mip_res5.cost[mip_res5.cost .< zero(eltype(mip_res5.cost))]))")
println("remaining cash: $(mip_res5.cash)")
println("used cash ≈ available cash: $(isapprox(sum(abs.(mip_res5.cost)) + mip_res5.cash, 4506.9 * sum(abs.(res5.w))))")

budget: 0.49999999995657685
long budget: 1.143177143743185
short budget: -0.643177143786608
weight bounds: true
┌────────┬─────────┬──────────┬─────────────┬─────────────┐
│ assets │  shares │     cost │ opt_weights │ mip_weights │
│ String │ Float64 │  Float64 │     Float64 │     Float64 │
├────────┼─────────┼──────────┼─────────────┼─────────────┤
│   AAPL │    -3.0 │ -377.022 │    -5.861 % │    -8.419 % │
│    AMD │     2.0 │   125.14 │     2.652 % │     2.771 % │
│    BAC │   -15.0 │ -484.515 │   -16.678 % │   -10.819 % │
│    BBY │     0.0 │      0.0 │     1.085 % │       0.0 % │
│    CVX │     2.0 │  347.456 │     9.537 % │     7.695 % │
│     GE │    -3.0 │ -191.649 │    -2.542 % │     -4.28 % │
│     HD │    -1.0 │  -311.22 │    -4.712 % │     -6.95 % │
│    JNJ │    12.0 │  2089.02 │    39.908 % │    46.262 % │
│    JPM │     7.0 │  907.025 │    19.622 % │    20.086 % │
│     KO │     8.0 │  500.872 │    10.332 % │    11.092 % │
│    LLY │    -1.0 │ -363.098 │    -4.609 % │   

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*